# Pandas Playground
Hands-on pandas patterns built from scratch plus a walk-through of `employees.csv`. Each block is concise so it is easy to reuse with other tables.


In [ ]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)


In [ ]:
demo_records = [
    {"region": "North", "product": "notebook", "units": 120, "margin": 0.28},
    {"region": "South", "product": "tablet", "units": 80, "margin": 0.31},
    {"region": "West", "product": "monitor", "units": 55, "margin": 0.24},
    {"region": "East", "product": "laptop", "units": 200, "margin": 0.34},
]
demo_df = pd.DataFrame(demo_records)
demo_df = demo_df.assign(revenue=lambda df: df.units * (1 + df.margin) * 100)
demo_df


In [ ]:
df = pd.read_csv('employees.csv')
print(f"rows: {len(df)}, columns: {df.columns.tolist()}")
df.sample(5, random_state=7)


In [ ]:
column_overview = pd.DataFrame({
    'dtype': df.dtypes,
    'missing': df.isna().sum(),
    'unique': df.nunique()
})
column_overview


In [ ]:
numerical_cols = df.select_dtypes(include=[np.number]).columns
summary = df[numerical_cols].describe().T
summary[['mean', 'std', 'min', 'max']]


In [ ]:
well_paid = (
    df.loc[df['Salary'] > df['Salary'].median(), ['Name', 'Country', 'Department', 'Salary']]
      .sort_values(['Department', 'Salary'], ascending=[True, False])
      .head(10)
)
well_paid


In [ ]:
salary_matrix = df.pivot_table(
    index='Department', columns='Country', values='Salary', aggfunc='mean'
)
salary_matrix


In [ ]:
clean_df = df.copy()
if 'Age' in clean_df:
    clean_df['Age'] = clean_df['Age'].fillna(clean_df['Age'].median())
if 'Salary' in clean_df:
    clean_df['Salary'] = clean_df['Salary'].fillna(clean_df['Salary'].mean())
for cat_col in ['Country', 'Department', 'Name']:
    if cat_col in clean_df:
        clean_df[cat_col] = clean_df[cat_col].fillna(clean_df[cat_col].mode().iloc[0])
clean_df.isna().sum()


In [ ]:
if 'Age' in clean_df:
    bins = [0, 25, 35, 50, 70, np.inf]
    labels = ['junior', 'early', 'mid', 'senior', 'veteran']
    clean_df['age_band'] = pd.cut(clean_df['Age'], bins=bins, labels=labels, right=False)
if 'Salary' in clean_df:
    clean_df['salary_k'] = (clean_df['Salary'] / 1000).round(1)
clean_df.head()


In [ ]:
age_profile = None
if {'age_band', 'Salary'}.issubset(clean_df.columns):
    age_profile = (
        clean_df.groupby('age_band')
        .agg(employees=('Name', 'count'), avg_salary=('Salary', 'mean'))
        .sort_values('avg_salary', ascending=False)
    )
age_profile


In [ ]:
clean_df.sort_values(['salary_k', 'Department'], ascending=[False, True]).head(8)


In [ ]:
def quick_report(frame):
    return pd.DataFrame({
        'non_null': frame.notna().sum(),
        'unique_values': frame.nunique()
    })
quick_report(clean_df[['Country', 'Department', 'age_band']])
